In [7]:
import os
import sys
import pandas as pd
from datetime import datetime

# Dynamically append project root to path to prevent any workspace import errors
sys.path.append("/Users/mac/quant-fund-fundamentals")

class FundamentalEngine:
    def __init__(self):
        # FIX 1: Safe localized import to completely prevent circular loading bugs
        from src.fundamentals.scrapers.nse_scraper import NSEScraper
        self.scraper = NSEScraper()
        
        # Define directory for data output storage
        self.output_dir = "/Users/mac/quant-fund-fundamentals/src/fundamentals/data"
        os.makedirs(self.output_dir, exist_ok=True)

    def update_all_corporate_actions(self, trading_universe: list) -> pd.DataFrame:
        """
        Loops through the entire trading universe, fetches historical corporate 
        actions per symbol defensively, and aggregates them into a master database.
        """
        print(f"=== Starting Batch Update for {len(trading_universe)} Symbols ===")
        all_actions = []

        for i, raw_symbol in enumerate(trading_universe, 1):
            cleaned_symbol = self.scraper.clean_symbol(raw_symbol)
            print(f"[{i}/{len(trading_universe)}] Syncing: {raw_symbol} -> {cleaned_symbol}...")
            
            # FIX 2: Defensive exception wrapper to prevent single failures from halting execution
            try:
                df = self.scraper.get_corporate_actions(cleaned_symbol)
                
                if not df.empty:
                    all_actions.append(df)
                    print(f"   ↳ Success: Found {len(df)} corporate action events.")
                else:
                    print(f"   ↳ Notice: No recent corporate actions logged for {cleaned_symbol}.")
                    
            except Exception as loop_error:
                # Log the crash for audit purposes, but keep processing the remaining universe
                print(f"   [CRITICAL WARNING] Failed to process data loop for {cleaned_symbol}: {loop_error}")
                continue
            
        # Compile collected fragments into a structured master table
        if all_actions:
            master_df = pd.concat(all_actions, ignore_index=True)
            
            output_path = os.path.join(self.output_dir, "master_corporate_actions.csv")
            master_df.to_csv(output_path, index=False)
            
            print(f"\n=== BATCH PIPELINE SUCCESS ===")
            print(f"Master corporate actions database saved to: {output_path}")
            return master_df
        else:
            print("\n=== BATCH PIPELINE FAILURE ===")
            print("No actionable fundamental events data could be retrieved across the array.")
            return pd.DataFrame()

if __name__ == "__main__":
    TRADING_UNIVERSE = [
        "RELIANCE26MAYFUT", "TCS26MAYFUT", "HDFCBANK26MAYFUT", "INFY26MAYFUT",
        "ICICIBANK26MAYFUT", "HINDUNILVR26MAYFUT", "SBIN26MAYFUT", "BAJFINANCE26MAYFUT",
        "BHARTIARTL26MAYFUT", "KOTAKBANK26MAYFUT", "LT26MAYFUT", "AXISBANK26MAYFUT",
        "WIPRO26MAYFUT", "HCLTECH26MAYFUT", "MARUTI26MAYFUT", "SUNPHARMA26MAYFUT",
        "TATAMOTORS26MAYFUT", "TATASTEEL26MAYFUT", "NTPC26MAYFUT", "POWERGRID26MAYFUT",
        "ULTRACEMCO26MAYFUT", "TECHM26MAYFUT", "BAJAJFINSV26MAYFUT", "TITAN26MAYFUT",
        "NESTLEIND26MAYFUT", "ADANIENT26MAYFUT", "ADANIPORTS26MAYFUT", "COALINDIA26MAYFUT",
        "ONGC26MAYFUT", "JSWSTEEL26MAYFUT", "GRASIM26MAYFUT", "INDUSINDBK26MAYFUT",
        "DIVISLAB26MAYFUT", "DRREDDY26MAYFUT", "CIPLA26MAYFUT", "EICHERMOT26MAYFUT",
        "HEROMOTOCO26MAYFUT", "APOLLOHOSP26MAYFUT", "TATACONSUM26MAYFUT", "BRITANNIA26MAYFUT",
        "BPCL26MAYFUT", "IOC26MAYFUT", "HINDALCO26MAYFUT", "VEDL26MAYFUT",
        "SAIL26MAYFUT", "PNB26MAYFUT", "BANKBARODA26MAYFUT", "CANBK26MAYFUT",
        "FEDERALBNK26MAYFUT", "IDFCFIRSTB26MAYFUT"
    ]
    
    engine = FundamentalEngine()
    master_table = engine.update_all_corporate_actions(TRADING_UNIVERSE)
    
    if not master_table.empty:
        print("\nPreview of Master Data System Output:")
        print(master_table.head(10))

=== Starting Batch Update for 5 Symbols ===
[1/5] Syncing: RELIANCE26MAYFUT -> RELIANCE...
Error fetching corporate actions mirror for RELIANCE: Too Many Requests. Rate limited. Try after a while.
   ↳ Notice: No corporate actions logged for RELIANCE.
   (Cooldown: Pausing for 2.28s to avoid rate limits...)
[2/5] Syncing: TCS26MAYFUT -> TCS...
Error fetching corporate actions mirror for TCS: Too Many Requests. Rate limited. Try after a while.
   ↳ Notice: No corporate actions logged for TCS.
   (Cooldown: Pausing for 2.59s to avoid rate limits...)
[3/5] Syncing: HDFCBANK26MAYFUT -> HDFCBANK...
Error fetching corporate actions mirror for HDFCBANK: Too Many Requests. Rate limited. Try after a while.
   ↳ Notice: No corporate actions logged for HDFCBANK.
   (Cooldown: Pausing for 1.50s to avoid rate limits...)
[4/5] Syncing: INFY26MAYFUT -> INFY...
Error fetching corporate actions mirror for INFY: Too Many Requests. Rate limited. Try after a while.
   ↳ Notice: No corporate actions logged